In [17]:
# ============================================================
# H8: LABEL-AWARE PROTOTYPE + SUPERVISED CONTRASTIVE
#     + ORDINAL HYPERNETWORK
#
# H8 builds directly on H7.
#
# H7:
#   support embeddings + labels
#            ↓
#   label-aware attention
#            ↓
#   student context
#            ↓
#   HyperNetwork
#            ↓
#   personalized classifier
#
# H8:
#   support embeddings + labels
#            ↓
#   label-aware support encoder
#            ├─────────────────────┐
#            ↓                     ↓
#   student context          class prototypes
#            ↓                     ↓
#       HyperNetwork       prototype similarity
#            ↓                     ↓
#            └──────────┬──────────┘
#                       ↓
#                 final logits
#
# TRAINING OBJECTIVE:
#
#   Total loss =
#       CE
#     + LAMBDA_CONTRASTIVE * SupervisedContrastiveLoss
#     + LAMBDA_ORDINAL     * OrdinalLoss
#
# NO:
#   focal loss
#   aggressive class weighting
#   artificial test balancing
#   fabricated minority examples
#
# TEST:
#
#   unseen student
#       ↓
#   first K REAL clips (early)
#       OR
#   K REAL random clips (random)
#       ↓
#   personalization
#       ↓
#   remaining REAL clips
#
# ============================================================

import json
import os
import re
import random
from collections import Counter

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    confusion_matrix,
)

from sklearn.preprocessing import StandardScaler


# ============================================================
# CONFIGURATION
# ============================================================

LABELS = [
    "Highly Disengage",
    "Disengage",
    "Engage",
    "Highly Engage",
]

LABEL_TO_ID = {
    label: i
    for i, label in enumerate(LABELS)
}

NUM_CLASSES = 4
INPUT_DIM = 1024

FEATURE_COLS = [
    f"emb_{i}"
    for i in range(INPUT_DIM)
]


# ============================================================
# MODEL DIMENSIONS
# ============================================================

EMBED_DIM = 128
CONTEXT_DIM = 128
LABEL_EMBED_DIM = 32

ATTENTION_HIDDEN = 128
HYPER_HIDDEN = 128


# ============================================================
# TRAINING
# ============================================================

DEFAULT_EPISODES = 2000
DEFAULT_SEED = 42

LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

QUERY_MAX = 64

TRAIN_K_VALUES = [5, 10, 20]


# ============================================================
# H8 LOSS WEIGHTS
#
# Keep these modest.
#
# CE remains the main objective.
# ============================================================

LAMBDA_CONTRASTIVE = 0.10
LAMBDA_ORDINAL = 0.05

CONTRASTIVE_TEMPERATURE = 0.10


# ============================================================
# PROTOTYPE CONTRIBUTION
#
# Final logits:
#
#   hyper_logits
#       +
#   PROTOTYPE_WEIGHT * prototype_logits
#
# Keep this relatively small initially so the prototype
# mechanism assists rather than completely replacing the
# HyperNetwork.
# ============================================================

PROTOTYPE_WEIGHT = 0.50


# ============================================================
# REPRODUCIBILITY
# ============================================================

def set_seed(seed):

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# ============================================================
# DATA LOADING
# ============================================================

def parse_group_from_clip_id(clip_id):

    """
    Extract student/group identifier.

    Expected structure similar to:

        video123_17_person4

    becomes:

        video123_person4
    """

    match = re.match(
        r"^(video[^_]+)_(\d+)_((?:person).+)$",
        str(clip_id)
    )

    if not match:
        return None

    return (
        f"{match.group(1)}_"
        f"{match.group(3)}"
    )


def load_clip_dataframe(json_path):

    with open(
        json_path,
        "r",
        encoding="utf-8"
    ) as f:

        data = json.load(f)

    records = []

    for clip_id, meta in data.items():

        if not isinstance(meta, dict):
            continue

        label = meta.get("label")

        if label not in LABELS:
            continue

        embeds = meta.get("embeds")

        if not isinstance(embeds, list):
            continue

        if len(embeds) != INPUT_DIM:
            continue

        group = parse_group_from_clip_id(
            clip_id
        )

        if group is None:
            continue

        arr = np.asarray(
            embeds,
            dtype=np.float32
        )

        if not np.all(np.isfinite(arr)):
            continue

        clip_match = re.search(
            r"_(\d+)_",
            str(clip_id)
        )

        clip_index = (
            int(clip_match.group(1))
            if clip_match
            else 0
        )

        records.append(
            {
                "clip_id": str(clip_id),
                "group": group,
                "label": str(label),
                "label_id": LABEL_TO_ID[
                    str(label)
                ],
                "clip_index": clip_index,
                "embedding": arr,
            }
        )

    if not records:

        raise ValueError(
            f"No valid labelled clips found "
            f"in {json_path}"
        )

    df = pd.DataFrame(records)

    emb_matrix = np.vstack(
        df["embedding"].to_numpy()
    )

    emb_df = pd.DataFrame(
        emb_matrix,
        columns=FEATURE_COLS,
        index=df.index,
    )

    df = pd.concat(
        [
            df.drop(
                columns=["embedding"]
            ),
            emb_df,
        ],
        axis=1,
    )

    return df


def load_group_fold_map(fold_csv_path):

    fold_df = pd.read_csv(
        fold_csv_path
    )

    required = {
        "group",
        "fold",
    }

    missing = (
        required
        - set(fold_df.columns)
    )

    if missing:

        raise ValueError(
            "Fold CSV missing required "
            f"columns: {sorted(missing)}"
        )

    group_fold = (
        fold_df[
            ["group", "fold"]
        ]
        .drop_duplicates()
    )

    check = (
        group_fold
        .groupby("group")["fold"]
        .nunique()
    )

    bad = check[
        check != 1
    ]

    if len(bad) > 0:

        raise ValueError(
            "Groups assigned to multiple "
            f"folds: {bad.index.tolist()}"
        )

    return dict(
        zip(
            group_fold["group"],
            group_fold["fold"].astype(int),
        )
    )


# ============================================================
# TEST SUPPORT / QUERY SAMPLING
#
# DO NOT CHANGE THIS FOR H8.
# ============================================================

def sample_support_query(
    student_df,
    k_shot,
    protocol,
    rng,
):

    ordered = (
        student_df
        .sort_values("clip_index")
        .copy()
    )

    if len(ordered) <= k_shot:

        return (
            None,
            None,
            f"insufficient_clips_n="
            f"{len(ordered)}_k={k_shot}",
        )

    if protocol == "early":

        # ACTUAL FIRST K REAL CLIPS
        support = ordered.iloc[
            :k_shot
        ].copy()

    elif protocol == "random":

        # ACTUAL RANDOM K REAL CLIPS
        sampled_idx = rng.choice(
            ordered.index.to_numpy(),
            size=k_shot,
            replace=False,
        )

        support = ordered.loc[
            sampled_idx
        ].copy()

    else:

        return (
            None,
            None,
            f"unknown_protocol_{protocol}",
        )

    query = ordered.loc[
        ~ordered.index.isin(
            support.index
        )
    ].copy()

    if query.empty:

        return (
            None,
            None,
            "empty_query",
        )

    return (
        support,
        query,
        None,
    )


# ============================================================
# H8 TRAINING SUPPORT SAMPLER
# ============================================================

def sample_balanced_support(
    student_df,
    k_shot,
    rng,
):

    available_classes = []
    class_indices = {}

    for class_id in range(NUM_CLASSES):

        idx = (
            student_df[
                student_df["label_id"]
                == class_id
            ]
            .index
            .to_numpy()
        )

        if len(idx) > 0:

            available_classes.append(
                class_id
            )

            class_indices[
                class_id
            ] = idx

    if not available_classes:
        return None

    n_classes = len(
        available_classes
    )

    base = k_shot // n_classes

    remainder = k_shot % n_classes

    allocation = {
        c: base
        for c in available_classes
    }

    shuffled = (
        available_classes.copy()
    )

    rng.shuffle(shuffled)

    for c in shuffled[:remainder]:

        allocation[c] += 1

    selected_indices = []

    unused_slots = 0

    for c in available_classes:

        requested = allocation[c]

        available = class_indices[c]

        take = min(
            requested,
            len(available)
        )

        if take < requested:

            unused_slots += (
                requested - take
            )

        chosen = rng.choice(
            available,
            size=take,
            replace=False,
        )

        selected_indices.extend(
            chosen.tolist()
        )

    # --------------------------------------------------------
    # Redistribute unused slots.
    # --------------------------------------------------------

    if unused_slots > 0:

        selected_set = set(
            selected_indices
        )

        remaining = []

        for c in available_classes:

            for idx in class_indices[c]:

                if idx not in selected_set:

                    remaining.append(idx)

        if remaining:

            take = min(
                unused_slots,
                len(remaining)
            )

            extra = rng.choice(
                np.asarray(remaining),
                size=take,
                replace=False,
            )

            selected_indices.extend(
                extra.tolist()
            )

    if not selected_indices:
        return None

    return student_df.loc[
        selected_indices
    ].copy()


# ============================================================
# H8 TRAINING QUERY SAMPLER
# ============================================================

def sample_balanced_query(
    query_df,
    max_samples,
    rng,
):

    if query_df.empty:
        return query_df

    available_classes = []
    class_indices = {}

    for class_id in range(NUM_CLASSES):

        idx = (
            query_df[
                query_df["label_id"]
                == class_id
            ]
            .index
            .to_numpy()
        )

        if len(idx) > 0:

            available_classes.append(
                class_id
            )

            class_indices[
                class_id
            ] = idx

    if not available_classes:

        return query_df.iloc[0:0]

    n_classes = len(
        available_classes
    )

    per_class = max_samples // n_classes

    selected = []

    for c in available_classes:

        idx = class_indices[c]

        take = min(
            per_class,
            len(idx)
        )

        if take > 0:

            chosen = rng.choice(
                idx,
                size=take,
                replace=False,
            )

            selected.extend(
                chosen.tolist()
            )

    # Fill remaining slots.
    selected_set = set(selected)

    remaining = []

    for c in available_classes:

        for idx in class_indices[c]:

            if idx not in selected_set:

                remaining.append(idx)

    slots = (
        max_samples
        - len(selected)
    )

    if slots > 0 and remaining:

        take = min(
            slots,
            len(remaining)
        )

        extra = rng.choice(
            np.asarray(remaining),
            size=take,
            replace=False,
        )

        selected.extend(
            extra.tolist()
        )

    return query_df.loc[
        selected
    ].copy()


# ============================================================
# LABEL-AWARE SUPPORT ENCODER
# ============================================================

class LabelAwareContextEncoder(
    nn.Module
):

    def __init__(self):

        super().__init__()

        # ----------------------------------------------------
        # Visual embedding encoder
        # ----------------------------------------------------

        self.embedding_encoder = nn.Sequential(

            nn.Linear(
                INPUT_DIM,
                256
            ),

            nn.ReLU(),

            nn.LayerNorm(256),

            nn.Linear(
                256,
                EMBED_DIM
            ),

            nn.ReLU(),

            nn.LayerNorm(
                EMBED_DIM
            ),
        )

        # ----------------------------------------------------
        # Explicit class label embedding
        # ----------------------------------------------------

        self.label_embedding = nn.Embedding(
            NUM_CLASSES,
            LABEL_EMBED_DIM
        )

        # ----------------------------------------------------
        # Label-aware fusion
        # ----------------------------------------------------

        self.support_fusion = nn.Sequential(

            nn.Linear(
                EMBED_DIM
                + LABEL_EMBED_DIM,
                CONTEXT_DIM
            ),

            nn.ReLU(),

            nn.LayerNorm(
                CONTEXT_DIM
            ),
        )

        # ----------------------------------------------------
        # Attention
        # ----------------------------------------------------

        self.attention = nn.Sequential(

            nn.Linear(
                CONTEXT_DIM,
                ATTENTION_HIDDEN
            ),

            nn.Tanh(),

            nn.Linear(
                ATTENTION_HIDDEN,
                1
            ),
        )

        # ----------------------------------------------------
        # Context projection
        # ----------------------------------------------------

        self.context_projection = nn.Sequential(

            nn.Linear(
                CONTEXT_DIM,
                CONTEXT_DIM
            ),

            nn.ReLU(),

            nn.LayerNorm(
                CONTEXT_DIM
            ),
        )

    def encode_visual(
        self,
        x
    ):

        return self.embedding_encoder(x)

    def forward(
        self,
        support_x,
        support_y,
    ):

        visual = self.embedding_encoder(
            support_x
        )

        label_emb = self.label_embedding(
            support_y
        )

        fused = torch.cat(
            [
                visual,
                label_emb,
            ],
            dim=1,
        )

        support_repr = (
            self.support_fusion(
                fused
            )
        )

        attention_logits = (
            self.attention(
                support_repr
            )
        )

        attention_weights = F.softmax(
            attention_logits,
            dim=0,
        )

        context = (
            attention_weights
            * support_repr
        ).sum(dim=0)

        context = (
            self.context_projection(
                context
            )
        )

        return (
            context,
            attention_weights.squeeze(-1),
            visual,
            support_repr,
        )


# ============================================================
# QUERY ENCODER
# ============================================================

class QueryEncoder(
    nn.Module
):

    def __init__(self):

        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(
                INPUT_DIM,
                256
            ),

            nn.ReLU(),

            nn.LayerNorm(256),

            nn.Linear(
                256,
                EMBED_DIM
            ),

            nn.ReLU(),

            nn.LayerNorm(
                EMBED_DIM
            ),
        )

    def forward(
        self,
        x
    ):

        return self.net(x)


# ============================================================
# HYPERNETWORK
# ============================================================

HEAD_PARAM_DIM = (
    NUM_CLASSES * EMBED_DIM
    + NUM_CLASSES
)


class HyperNetwork(
    nn.Module
):

    def __init__(self):

        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(
                CONTEXT_DIM,
                HYPER_HIDDEN
            ),

            nn.ReLU(),

            nn.LayerNorm(
                HYPER_HIDDEN
            ),

            nn.Linear(
                HYPER_HIDDEN,
                HYPER_HIDDEN
            ),

            nn.ReLU(),
        )

        self.output = nn.Linear(
            HYPER_HIDDEN,
            HEAD_PARAM_DIM
        )

    def forward(
        self,
        context
    ):

        params = self.output(
            self.net(context)
        )

        weight_end = (
            NUM_CLASSES
            * EMBED_DIM
        )

        weight = (
            params[
                :weight_end
            ]
            .view(
                NUM_CLASSES,
                EMBED_DIM
            )
        )

        bias = params[
            weight_end:
        ]

        return (
            weight,
            bias,
        )


# ============================================================
# PROTOTYPE CONSTRUCTION
# ============================================================

def build_class_prototypes(
    support_visual,
    support_y,
):

    """
    Construct one prototype for every class that is actually
    represented in the support set.

    IMPORTANT:

    Missing classes are NOT fabricated.

    The returned mask tells the classifier which prototypes
    are genuinely supported by the student's support set.
    """

    prototypes = []

    available_mask = []

    for class_id in range(NUM_CLASSES):

        mask = (
            support_y
            == class_id
        )

        if mask.any():

            prototype = (
                support_visual[mask]
                .mean(dim=0)
            )

            prototype = F.normalize(
                prototype,
                p=2,
                dim=0,
            )

            available_mask.append(
                True
            )

        else:

            # Placeholder only.
            # It will NOT contribute to prototype logits.
            prototype = torch.zeros(
                EMBED_DIM,
                device=support_visual.device,
            )

            available_mask.append(
                False
            )

        prototypes.append(
            prototype
        )

    prototypes = torch.stack(
        prototypes,
        dim=0,
    )

    available_mask = torch.tensor(
        available_mask,
        dtype=torch.bool,
        device=support_visual.device,
    )

    return (
        prototypes,
        available_mask,
    )


# ============================================================
# PROTOTYPE LOGITS
# ============================================================

def prototype_logits(
    query_embeddings,
    prototypes,
    available_mask,
):

    query_norm = F.normalize(
        query_embeddings,
        p=2,
        dim=1,
    )

    logits = (
        query_norm
        @ prototypes.T
    )

    # --------------------------------------------------------
    # Missing classes have no student-specific prototype.
    #
    # We therefore neutralize those prototype logits rather
    # than pretending that the student provided evidence.
    # --------------------------------------------------------

    logits = logits.masked_fill(
        ~available_mask.unsqueeze(0),
        0.0,
    )

    return logits


# ============================================================
# H8 FULL MODEL
# ============================================================

class H8Model(
    nn.Module
):

    def __init__(self):

        super().__init__()

        self.context_encoder = (
            LabelAwareContextEncoder()
        )

        self.query_encoder = (
            QueryEncoder()
        )

        self.hypernetwork = (
            HyperNetwork()
        )

    def forward(
        self,
        support_x,
        support_y,
        query_x,
    ):

        # ----------------------------------------------------
        # Label-aware student context
        # ----------------------------------------------------

        (
            context,
            attention,
            support_visual,
            support_repr,
        ) = self.context_encoder(
            support_x,
            support_y,
        )

        # ----------------------------------------------------
        # Personalized HyperNetwork head
        # ----------------------------------------------------

        weight, bias = (
            self.hypernetwork(
                context
            )
        )

        # ----------------------------------------------------
        # Query representation
        # ----------------------------------------------------

        query_embeddings = (
            self.query_encoder(
                query_x
            )
        )

        # ----------------------------------------------------
        # HyperNetwork classifier
        # ----------------------------------------------------

        hyper_logits = F.linear(
            query_embeddings,
            weight,
            bias,
        )

        # ----------------------------------------------------
        # Class prototypes
        # ----------------------------------------------------

        (
            prototypes,
            available_mask,
        ) = build_class_prototypes(
            support_visual,
            support_y,
        )

        # ----------------------------------------------------
        # Prototype similarity
        # ----------------------------------------------------

        proto_logits = prototype_logits(
            query_embeddings,
            prototypes,
            available_mask,
        )

        # ----------------------------------------------------
        # Combined classifier
        # ----------------------------------------------------

        final_logits = (
            hyper_logits
            +
            PROTOTYPE_WEIGHT
            * proto_logits
        )

        return {
            "logits":
                final_logits,

            "hyper_logits":
                hyper_logits,

            "prototype_logits":
                proto_logits,

            "query_embeddings":
                query_embeddings,

            "support_visual":
                support_visual,

            "support_repr":
                support_repr,

            "prototypes":
                prototypes,

            "prototype_mask":
                available_mask,

            "context":
                context,

            "attention":
                attention,
        }


# ============================================================
# SUPERVISED CONTRASTIVE LOSS
# ============================================================

def supervised_contrastive_loss(
    embeddings,
    labels,
    temperature=CONTRASTIVE_TEMPERATURE,
):

    """
    Supervised contrastive learning.

    Same-label examples:
        pull together.

    Different-label examples:
        push apart.

    This is applied to query embeddings during meta-training.

    If a batch does not contain any positive pair, the loss
    safely becomes zero.
    """

    if embeddings.size(0) < 2:

        return embeddings.new_tensor(
            0.0
        )

    embeddings = F.normalize(
        embeddings,
        p=2,
        dim=1,
    )

    similarity = (
        embeddings
        @ embeddings.T
    )

    similarity = (
        similarity
        / temperature
    )

    batch_size = (
        embeddings.size(0)
    )

    # Remove self-comparisons.
    self_mask = torch.eye(
        batch_size,
        dtype=torch.bool,
        device=embeddings.device,
    )

    similarity = similarity.masked_fill(
        self_mask,
        -1e9,
    )

    labels = labels.view(-1, 1)

    positive_mask = (
        torch.eq(
            labels,
            labels.T,
        )
    )

    positive_mask = positive_mask & (
        ~self_mask
    )

    log_prob = (
        similarity
        -
        torch.logsumexp(
            similarity,
            dim=1,
            keepdim=True,
        )
    )

    positive_counts = (
        positive_mask.sum(dim=1)
    )

    valid = (
        positive_counts > 0
    )

    if not valid.any():

        return embeddings.new_tensor(
            0.0
        )

    mean_log_prob_pos = (
        (
            log_prob
            * positive_mask.float()
        )
        .sum(dim=1)
        /
        positive_counts.clamp(
            min=1
        )
    )

    loss = -mean_log_prob_pos[
        valid
    ].mean()

    return loss


# ============================================================
# ORDINAL LOSS
# ============================================================

def ordinal_loss(
    logits,
    targets,
):

    """
    The classes have an intrinsic ordering:

        0 Highly Disengage
        1 Disengage
        2 Engage
        3 Highly Engage

    Standard CE treats every wrong class as categorically
    different.

    This auxiliary loss penalizes large ordinal errors more.

    Expected class:
        sum(probability_i * class_index_i)

    Target:
        class_index

    We normalize by NUM_CLASSES - 1 so the magnitude remains
    small.
    """

    probabilities = F.softmax(
        logits,
        dim=1,
    )

    class_indices = torch.arange(
        NUM_CLASSES,
        device=logits.device,
        dtype=torch.float32,
    )

    expected_class = (
        probabilities
        * class_indices.unsqueeze(0)
    ).sum(dim=1)

    target_float = (
        targets.float()
    )

    normalized_error = (
        expected_class
        - target_float
    ) / (
        NUM_CLASSES - 1
    )

    return (
        normalized_error ** 2
    ).mean()


# ============================================================
# H8 TOTAL LOSS
# ============================================================

def h8_loss(
    outputs,
    targets,
):

    # --------------------------------------------------------
    # Main classification objective
    # --------------------------------------------------------

    ce = F.cross_entropy(
        outputs["logits"],
        targets,
    )

    # --------------------------------------------------------
    # Representation separation
    # --------------------------------------------------------

    contrastive = (
        supervised_contrastive_loss(
            outputs["query_embeddings"],
            targets,
        )
    )

    # --------------------------------------------------------
    # Ordinal structure
    # --------------------------------------------------------

    ordinal = ordinal_loss(
        outputs["logits"],
        targets,
    )

    # --------------------------------------------------------
    # Combined objective
    # --------------------------------------------------------

    total = (
        ce
        +
        LAMBDA_CONTRASTIVE
        * contrastive
        +
        LAMBDA_ORDINAL
        * ordinal
    )

    return (
        total,
        ce,
        contrastive,
        ordinal,
    )


# ============================================================
# H8 EPISODIC META-TRAINING
# ============================================================

def episodic_train(
    model,
    train_df,
    scaler,
    episodes,
    seed,
    device,
):

    model.train()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    rng = np.random.default_rng(
        seed
    )

    groups = sorted(
        train_df["group"].unique()
    )

    successful = 0

    running_losses = []
    running_ce = []
    running_contrastive = []
    running_ordinal = []

    for episode in range(
        episodes
    ):

        # ----------------------------------------------------
        # Select training student
        # ----------------------------------------------------

        group = rng.choice(
            groups
        )

        student_df = (
            train_df[
                train_df["group"]
                == group
            ]
            .copy()
        )

        k = int(
            rng.choice(
                TRAIN_K_VALUES
            )
        )

        if len(student_df) <= k:

            continue

        # ----------------------------------------------------
        # CLASS-AWARE TRAINING SUPPORT
        # ----------------------------------------------------

        support_df = (
            sample_balanced_support(
                student_df,
                k,
                rng,
            )
        )

        if (
            support_df is None
            or support_df.empty
        ):

            continue

        # ----------------------------------------------------
        # Remaining examples
        # ----------------------------------------------------

        query_df = student_df.loc[
            ~student_df.index.isin(
                support_df.index
            )
        ].copy()

        if query_df.empty:
            continue

        # ----------------------------------------------------
        # CLASS-AWARE TRAINING QUERY
        # ----------------------------------------------------

        query_df = (
            sample_balanced_query(
                query_df,
                QUERY_MAX,
                rng,
            )
        )

        if query_df.empty:
            continue

        # ----------------------------------------------------
        # Scale
        # ----------------------------------------------------

        support_x_np = scaler.transform(
            support_df[
                FEATURE_COLS
            ].to_numpy(
                dtype=np.float32
            )
        )

        query_x_np = scaler.transform(
            query_df[
                FEATURE_COLS
            ].to_numpy(
                dtype=np.float32
            )
        )

        support_y_np = (
            support_df[
                "label_id"
            ]
            .to_numpy(
                dtype=np.int64
            )
        )

        query_y_np = (
            query_df[
                "label_id"
            ]
            .to_numpy(
                dtype=np.int64
            )
        )

        support_x = torch.tensor(
            support_x_np,
            dtype=torch.float32,
            device=device,
        )

        support_y = torch.tensor(
            support_y_np,
            dtype=torch.long,
            device=device,
        )

        query_x = torch.tensor(
            query_x_np,
            dtype=torch.float32,
            device=device,
        )

        query_y = torch.tensor(
            query_y_np,
            dtype=torch.long,
            device=device,
        )

        # ----------------------------------------------------
        # Forward
        # ----------------------------------------------------

        outputs = model(
            support_x,
            support_y,
            query_x,
        )

        # ----------------------------------------------------
        # H8 loss
        # ----------------------------------------------------

        (
            loss,
            ce,
            contrastive,
            ordinal,
        ) = h8_loss(
            outputs,
            query_y,
        )

        # ----------------------------------------------------
        # Backprop
        # ----------------------------------------------------

        optimizer.zero_grad()

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=5.0,
        )

        optimizer.step()

        successful += 1

        running_losses.append(
            loss.item()
        )

        running_ce.append(
            ce.item()
        )

        running_contrastive.append(
            contrastive.item()
        )

        running_ordinal.append(
            ordinal.item()
        )

        # ----------------------------------------------------
        # Logging
        # ----------------------------------------------------

        if (
            (episode + 1) % 100
            == 0
        ):

            print(
                f"    Episode "
                f"{episode + 1}/{episodes} | "
                f"total={np.mean(running_losses[-100:]):.4f} | "
                f"CE={np.mean(running_ce[-100:]):.4f} | "
                f"SupCon={np.mean(running_contrastive[-100:]):.4f} | "
                f"Ordinal={np.mean(running_ordinal[-100:]):.4f}"
            )

    print(
        f"    Successful episodes: "
        f"{successful}/{episodes}"
    )

    return model


# ============================================================
# METRICS
# ============================================================

def calculate_metrics(
    y_true,
    y_pred,
):

    accuracy = accuracy_score(
        y_true,
        y_pred,
    )

    balanced_accuracy = (
        balanced_accuracy_score(
            y_true,
            y_pred,
        )
    )

    macro_f1 = f1_score(
        y_true,
        y_pred,
        labels=list(
            range(NUM_CLASSES)
        ),
        average="macro",
        zero_division=0,
    )

    precision, recall, f1, support = (
        precision_recall_fscore_support(
            y_true,
            y_pred,
            labels=list(
                range(NUM_CLASSES)
            ),
            zero_division=0,
        )
    )

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=list(
            range(NUM_CLASSES)
        ),
    )

    return {
        "accuracy":
            float(accuracy),

        "balanced_accuracy":
            float(balanced_accuracy),

        "macro_f1":
            float(macro_f1),

        "precision":
            precision,

        "recall":
            recall,

        "f1":
            f1,

        "support":
            support,

        "confusion_matrix":
            cm,
    }


# ============================================================
# MAJORITY BASELINE
# ============================================================

def majority_prediction(
    y_query,
):

    counts = Counter(
        y_query.tolist()
    )

    majority_class = max(
        counts,
        key=counts.get,
    )

    return np.full(
        len(y_query),
        majority_class,
        dtype=np.int64,
    )


# ============================================================
# EVALUATION
# ============================================================

def evaluate_student(
    model,
    support_df,
    query_df,
    scaler,
    device,
):

    # --------------------------------------------------------
    # Transform
    # --------------------------------------------------------

    support_x_np = scaler.transform(
        support_df[
            FEATURE_COLS
        ].to_numpy(
            dtype=np.float32
        )
    )

    query_x_np = scaler.transform(
        query_df[
            FEATURE_COLS
        ].to_numpy(
            dtype=np.float32
        )
    )

    support_y_np = (
        support_df[
            "label_id"
        ]
        .to_numpy(
            dtype=np.int64
        )
    )

    y_true = (
        query_df[
            "label_id"
        ]
        .to_numpy(
            dtype=np.int64
        )
    )

    support_x = torch.tensor(
        support_x_np,
        dtype=torch.float32,
        device=device,
    )

    support_y = torch.tensor(
        support_y_np,
        dtype=torch.long,
        device=device,
    )

    query_x = torch.tensor(
        query_x_np,
        dtype=torch.float32,
        device=device,
    )

    # --------------------------------------------------------
    # Prediction
    # --------------------------------------------------------

    model.eval()

    with torch.no_grad():

        outputs = model(
            support_x,
            support_y,
            query_x,
        )

        logits = outputs[
            "logits"
        ]

        y_pred = (
            torch.argmax(
                logits,
                dim=1,
            )
            .cpu()
            .numpy()
        )

    # --------------------------------------------------------
    # Model metrics
    # --------------------------------------------------------

    metrics = calculate_metrics(
        y_true,
        y_pred,
    )

    # --------------------------------------------------------
    # Majority baseline
    # --------------------------------------------------------

    majority_pred = (
        majority_prediction(
            y_true
        )
    )

    majority_metrics = (
        calculate_metrics(
            y_true,
            majority_pred,
        )
    )

    # --------------------------------------------------------
    # Additional H8 diagnostic:
    # Was the prototype mechanism actually used?
    # --------------------------------------------------------

    prototype_mask = (
        outputs[
            "prototype_mask"
        ]
        .cpu()
        .numpy()
    )

    prototype_available_count = int(
        prototype_mask.sum()
    )

    return (
        y_true,
        y_pred,
        metrics,
        majority_metrics,
        prototype_available_count,
        prototype_mask,
    )


# ============================================================
# RUN EXPERIMENT
# ============================================================

def run_experiment(
    clip_df,
    group_fold,
    k_values,
    seed,
    episodes,
    device,
    output_dir,
):

    eligible_groups = set(
        group_fold.keys()
    )

    eval_df = clip_df[
        clip_df["group"].isin(
            eligible_groups
        )
    ].copy()

    summary_rows = []
    per_student_rows = []
    per_class_rows = []
    confusion_rows = []
    skipped_rows = []

    # ========================================================
    # OUTER FOLDS
    # ========================================================

    fold_ids = sorted(
        set(
            group_fold.values()
        )
    )

    for fold_id in fold_ids:

        print()
        print("=" * 80)
        print(
            f"H8 OUTER FOLD {fold_id}"
        )
        print("=" * 80)

        test_groups = [
            g
            for g, f in group_fold.items()
            if f == fold_id
        ]

        train_groups = [
            g
            for g, f in group_fold.items()
            if f != fold_id
        ]

        train_df = eval_df[
            eval_df["group"].isin(
                train_groups
            )
        ].copy()

        test_df = eval_df[
            eval_df["group"].isin(
                test_groups
            )
        ].copy()

        # ----------------------------------------------------
        # Leakage check
        # ----------------------------------------------------

        overlap = (
            set(train_df["group"])
            &
            set(test_df["group"])
        )

        if overlap:

            raise RuntimeError(
                f"DATA LEAKAGE in fold "
                f"{fold_id}: {overlap}"
            )

        print(
            f"Training students: "
            f"{train_df['group'].nunique()}"
        )

        print(
            f"Test students: "
            f"{test_df['group'].nunique()}"
        )

        # ----------------------------------------------------
        # Training class distribution
        # ----------------------------------------------------

        print()
        print(
            "Training class distribution:"
        )

        print(
            train_df[
                "label"
            ].value_counts()
        )

        # ----------------------------------------------------
        # Fit scaler ONLY on training data
        # ----------------------------------------------------

        scaler = StandardScaler()

        scaler.fit(
            train_df[
                FEATURE_COLS
            ].to_numpy(
                dtype=np.float32
            )
        )

        # ----------------------------------------------------
        # Fresh model for every fold
        # ----------------------------------------------------

        set_seed(
            seed + fold_id
        )

        model = H8Model()

        model.to(device)

        # ----------------------------------------------------
        # Meta-training
        # ----------------------------------------------------

        print()
        print(
            f"H8 meta-training "
            f"({episodes} episodes)..."
        )

        episodic_train(
            model=model,
            train_df=train_df,
            scaler=scaler,
            episodes=episodes,
            seed=(
                seed
                + fold_id * 1000
            ),
            device=device,
        )

        # ====================================================
        # TEST STUDENTS
        # ====================================================

        for group in test_groups:

            student_df = test_df[
                test_df["group"]
                == group
            ].copy()

            if student_df.empty:
                continue

            for k in k_values:

                for protocol in [
                    "early",
                    "random",
                ]:

                    local_seed = (
                        seed
                        + fold_id * 10000
                        + k * 100
                        + sum(
                            ord(ch)
                            for ch in group
                        )
                        + (
                            1
                            if protocol
                            == "early"
                            else 0
                        )
                    )

                    rng = (
                        np.random.default_rng(
                            local_seed
                        )
                    )

                    # ------------------------------------------------
                    # REAL TEST SUPPORT
                    # ------------------------------------------------

                    (
                        support_df,
                        query_df,
                        reason,
                    ) = (
                        sample_support_query(
                            student_df,
                            k,
                            protocol,
                            rng,
                        )
                    )

                    if reason is not None:

                        skipped_rows.append(
                            {
                                "outer_fold":
                                    fold_id,

                                "student_group":
                                    group,

                                "k_shot":
                                    k,

                                "support_protocol":
                                    protocol,

                                "reason":
                                    reason,
                            }
                        )

                        continue

                    # ------------------------------------------------
                    # EVALUATE
                    # ------------------------------------------------

                    (
                        y_true,
                        y_pred,
                        metrics,
                        majority_metrics,
                        prototype_count,
                        prototype_mask,
                    ) = evaluate_student(
                        model,
                        support_df,
                        query_df,
                        scaler,
                        device,
                    )

                    # ------------------------------------------------
                    # Per-student result
                    # ------------------------------------------------

                    row = {

                        "method":
                            "H8_label_aware_prototype_supcon_ordinal",

                        "outer_fold":
                            fold_id,

                        "student_group":
                            group,

                        "k_shot":
                            k,

                        "support_protocol":
                            protocol,

                        "support_size":
                            len(support_df),

                        "query_size":
                            len(query_df),

                        "support_highly_disengage":
                            int(
                                (
                                    support_df[
                                        "label_id"
                                    ]
                                    == 0
                                ).sum()
                            ),

                        "support_disengage":
                            int(
                                (
                                    support_df[
                                        "label_id"
                                    ]
                                    == 1
                                ).sum()
                            ),

                        "support_engage":
                            int(
                                (
                                    support_df[
                                        "label_id"
                                    ]
                                    == 2
                                ).sum()
                            ),

                        "support_highly_engage":
                            int(
                                (
                                    support_df[
                                        "label_id"
                                    ]
                                    == 3
                                ).sum()
                            ),

                        "prototype_classes_available":
                            prototype_count,

                        "accuracy":
                            metrics[
                                "accuracy"
                            ],

                        "balanced_accuracy":
                            metrics[
                                "balanced_accuracy"
                            ],

                        "macro_f1":
                            metrics[
                                "macro_f1"
                            ],

                        "majority_accuracy":
                            majority_metrics[
                                "accuracy"
                            ],

                        "majority_balanced_accuracy":
                            majority_metrics[
                                "balanced_accuracy"
                            ],

                        "majority_macro_f1":
                            majority_metrics[
                                "macro_f1"
                            ],
                    }

                    per_student_rows.append(
                        row
                    )

                    # ------------------------------------------------
                    # Per-class results
                    # ------------------------------------------------

                    for class_id, label in (
                        enumerate(LABELS)
                    ):

                        per_class_rows.append(
                            {

                                "method":
                                    "H8_label_aware_prototype_supcon_ordinal",

                                "outer_fold":
                                    fold_id,

                                "student_group":
                                    group,

                                "k_shot":
                                    k,

                                "support_protocol":
                                    protocol,

                                "class":
                                    label,

                                "precision":
                                    float(
                                        metrics[
                                            "precision"
                                        ][class_id]
                                    ),

                                "recall":
                                    float(
                                        metrics[
                                            "recall"
                                        ][class_id]
                                    ),

                                "f1":
                                    float(
                                        metrics[
                                            "f1"
                                        ][class_id]
                                    ),

                                "support":
                                    int(
                                        metrics[
                                            "support"
                                        ][class_id]
                                    ),
                            }
                        )

                    # ------------------------------------------------
                    # Confusion matrix
                    # ------------------------------------------------

                    cm = metrics[
                        "confusion_matrix"
                    ]

                    for true_id in range(
                        NUM_CLASSES
                    ):

                        for pred_id in range(
                            NUM_CLASSES
                        ):

                            confusion_rows.append(
                                {

                                    "method":
                                        "H8_label_aware_prototype_supcon_ordinal",

                                    "outer_fold":
                                        fold_id,

                                    "student_group":
                                        group,

                                    "k_shot":
                                        k,

                                    "support_protocol":
                                        protocol,

                                    "true_class":
                                        LABELS[
                                            true_id
                                        ],

                                    "predicted_class":
                                        LABELS[
                                            pred_id
                                        ],

                                    "count":
                                        int(
                                            cm[
                                                true_id,
                                                pred_id
                                            ]
                                        ),
                                }
                            )

    # ========================================================
    # DATAFRAMES
    # ========================================================

    per_student_df = pd.DataFrame(
        per_student_rows
    )

    per_class_df = pd.DataFrame(
        per_class_rows
    )

    confusion_df = pd.DataFrame(
        confusion_rows
    )

    skipped_df = pd.DataFrame(
        skipped_rows
    )

    if per_student_df.empty:

        raise RuntimeError(
            "No student evaluations "
            "were produced."
        )

    # ========================================================
    # SUMMARY
    # ========================================================

    summary_df = (
        per_student_df
        .groupby(
            [
                "support_protocol",
                "k_shot",
            ],
            as_index=False,
        )
        .agg(

            n_students=(
                "student_group",
                "nunique",
            ),

            total_query_clips=(
                "query_size",
                "sum",
            ),

            mean_accuracy=(
                "accuracy",
                "mean",
            ),

            sd_accuracy=(
                "accuracy",
                "std",
            ),

            mean_balanced_accuracy=(
                "balanced_accuracy",
                "mean",
            ),

            sd_balanced_accuracy=(
                "balanced_accuracy",
                "std",
            ),

            mean_macro_f1=(
                "macro_f1",
                "mean",
            ),

            sd_macro_f1=(
                "macro_f1",
                "std",
            ),

            mean_majority_accuracy=(
                "majority_accuracy",
                "mean",
            ),

            mean_majority_balanced_accuracy=(
                "majority_balanced_accuracy",
                "mean",
            ),

            mean_majority_macro_f1=(
                "majority_macro_f1",
                "mean",
            ),

            mean_prototype_classes_available=(
                "prototype_classes_available",
                "mean",
            ),
        )
        .sort_values(
            [
                "support_protocol",
                "k_shot",
            ]
        )
        .reset_index(
            drop=True
        )
    )

    # ========================================================
    # CLASS SUMMARY
    # ========================================================

    class_summary = (
        per_class_df
        .groupby(
            [
                "support_protocol",
                "k_shot",
                "class",
            ],
            as_index=False,
        )
        .agg(

            mean_precision=(
                "precision",
                "mean",
            ),

            mean_recall=(
                "recall",
                "mean",
            ),

            mean_f1=(
                "f1",
                "mean",
            ),

            total_support=(
                "support",
                "sum",
            ),
        )
    )

    # ========================================================
    # SAVE
    # ========================================================

    os.makedirs(
        output_dir,
        exist_ok=True,
    )

    per_student_path = os.path.join(
        output_dir,
        "h8_per_student.csv",
    )

    summary_path = os.path.join(
        output_dir,
        "h8_summary.csv",
    )

    per_class_path = os.path.join(
        output_dir,
        "h8_per_class.csv",
    )

    confusion_path = os.path.join(
        output_dir,
        "h8_confusion_matrix.csv",
    )

    skipped_path = os.path.join(
        output_dir,
        "h8_skipped.csv",
    )

    per_student_df.to_csv(
        per_student_path,
        index=False,
    )

    summary_df.to_csv(
        summary_path,
        index=False,
    )

    per_class_df.to_csv(
        per_class_path,
        index=False,
    )

    confusion_df.to_csv(
        confusion_path,
        index=False,
    )

    skipped_df.to_csv(
        skipped_path,
        index=False,
    )

    # ========================================================
    # PRINT RESULTS
    # ========================================================

    print()
    print("=" * 80)
    print(
        "H8: LABEL-AWARE PROTOTYPE + "
        "SUPERVISED CONTRASTIVE + "
        "ORDINAL HYPERNETWORK RESULTS"
    )
    print("=" * 80)

    print(
        summary_df.to_string(
            index=False
        )
    )

    # ========================================================
    # PER CLASS
    # ========================================================

    print()
    print("=" * 80)
    print(
        "H8 PER-CLASS RESULTS"
    )
    print("=" * 80)

    print(
        class_summary.to_string(
            index=False
        )
    )

    # ========================================================
    # SKIPPED
    # ========================================================

    print()
    print("=" * 80)
    print(
        "H8 SKIPPED CASES"
    )
    print("=" * 80)

    if skipped_df.empty:

        print("None.")

    else:

        print(
            skipped_df[
                "reason"
            ].value_counts()
        )

    # ========================================================
    # SAVED FILES
    # ========================================================

    print()
    print("=" * 80)
    print("FILES SAVED")
    print("=" * 80)

    print(per_student_path)
    print(summary_path)
    print(per_class_path)
    print(confusion_path)
    print(skipped_path)

    return (
        per_student_df,
        summary_df,
        per_class_df,
        confusion_df,
        skipped_df,
    )


# ============================================================
# COLAB ENTRY POINT
# ============================================================

if __name__ == "__main__":

    # --------------------------------------------------------
    # PATHS
    # --------------------------------------------------------

    JSON_PATH = (
        "/content/final_data_1.json"
    )

    FOLDS_PATH = (
        "/content/"
        "cmose_h1_h2_fold_protocol.csv"
    )

    OUTPUT_DIR = (
        "/content/h8_results"
    )

    # --------------------------------------------------------
    # EXPERIMENT SETTINGS
    # --------------------------------------------------------

    K_VALUES = [
        5,
        10,
        20,
    ]

    EPISODES = 2000

    SEED = 42

    # --------------------------------------------------------
    # DEVICE
    # --------------------------------------------------------

    if torch.cuda.is_available():

        DEVICE = torch.device(
            "cuda"
        )

        print(
            "CUDA available: True"
        )

        print(
            "GPU:",
            torch.cuda.get_device_name(0)
        )

    else:

        DEVICE = torch.device(
            "cpu"
        )

        print(
            "CUDA available: False"
        )

        print(
            "Using CPU."
        )

    # --------------------------------------------------------
    # REPRODUCIBILITY
    # --------------------------------------------------------

    set_seed(
        SEED
    )

    # ========================================================
    # LOAD DATA
    # ========================================================

    print()
    print(
        "Loading clip data..."
    )

    clip_df = (
        load_clip_dataframe(
            JSON_PATH
        )
    )

    print(
        f"Loaded "
        f"{len(clip_df)} labelled clips "
        f"from "
        f"{clip_df['group'].nunique()} "
        f"students."
    )

    print()
    print(
        "Overall class distribution:"
    )

    print(
        clip_df[
            "label"
        ].value_counts()
    )

    # ========================================================
    # LOAD FOLD PROTOCOL
    # ========================================================

    print()
    print(
        "Loading fold protocol..."
    )

    group_fold = (
        load_group_fold_map(
            FOLDS_PATH
        )
    )

    print(
        f"Fold protocol contains "
        f"{len(group_fold)} students."
    )

    # ========================================================
    # RUN H8
    # ========================================================

    run_experiment(
        clip_df=clip_df,
        group_fold=group_fold,
        k_values=K_VALUES,
        seed=SEED,
        episodes=EPISODES,
        device=DEVICE,
        output_dir=OUTPUT_DIR,
    )

CUDA available: True
GPU: Tesla T4

Loading clip data...
Loaded 11902 labelled clips from 103 students.

Overall class distribution:
label
Engage              8245
Disengage           2170
Highly Engage       1145
Highly Disengage     342
Name: count, dtype: int64

Loading fold protocol...
Fold protocol contains 82 students.

H8 OUTER FOLD 1
Training students: 65
Test students: 17

Training class distribution:
label
Engage              6291
Disengage           1601
Highly Engage        904
Highly Disengage     263
Name: count, dtype: int64

H8 meta-training (2000 episodes)...
    Episode 100/2000 | total=2.2247 | CE=1.7246 | SupCon=4.9647 | Ordinal=0.0719
    Episode 200/2000 | total=1.2894 | CE=0.8530 | SupCon=4.3459 | Ordinal=0.0362
    Episode 300/2000 | total=1.1300 | CE=0.7117 | SupCon=4.1677 | Ordinal=0.0310
    Episode 400/2000 | total=1.1411 | CE=0.7272 | SupCon=4.1223 | Ordinal=0.0330
    Episode 500/2000 | total=0.9813 | CE=0.5717 | SupCon=4.0831 | Ordinal=0.0254
    Episode 

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics


H8 OUTER FOLD 2
Training students: 65
Test students: 17

Training class distribution:
label
Engage              6415
Disengage           1605
Highly Engage        893
Highly Disengage     259
Name: count, dtype: int64

H8 meta-training (2000 episodes)...
    Episode 100/2000 | total=2.2737 | CE=1.7609 | SupCon=5.0914 | Ordinal=0.0734
    Episode 200/2000 | total=1.2229 | CE=0.7843 | SupCon=4.3696 | Ordinal=0.0331
    Episode 300/2000 | total=1.1163 | CE=0.6970 | SupCon=4.1763 | Ordinal=0.0320
    Episode 400/2000 | total=1.0775 | CE=0.6593 | SupCon=4.1675 | Ordinal=0.0297
    Episode 500/2000 | total=1.0261 | CE=0.6142 | SupCon=4.1055 | Ordinal=0.0272
    Episode 600/2000 | total=0.9548 | CE=0.5493 | SupCon=4.0442 | Ordinal=0.0223
    Episode 700/2000 | total=0.9253 | CE=0.5189 | SupCon=4.0527 | Ordinal=0.0215
    Episode 800/2000 | total=0.8889 | CE=0.4899 | SupCon=3.9804 | Ordinal=0.0197
    Episode 900/2000 | total=0.8926 | CE=0.4917 | SupCon=3.9990 | Ordinal=0.0206
    Episode 100

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics


H8 OUTER FOLD 3
Training students: 66
Test students: 16

Training class distribution:
label
Engage              6459
Disengage           1628
Highly Engage        872
Highly Disengage     251
Name: count, dtype: int64

H8 meta-training (2000 episodes)...
    Episode 100/2000 | total=2.2277 | CE=1.7100 | SupCon=5.1446 | Ordinal=0.0651
    Episode 200/2000 | total=1.2867 | CE=0.8464 | SupCon=4.3834 | Ordinal=0.0391
    Episode 300/2000 | total=1.1474 | CE=0.7168 | SupCon=4.2898 | Ordinal=0.0324
    Episode 400/2000 | total=1.0993 | CE=0.6812 | SupCon=4.1660 | Ordinal=0.0313
    Episode 500/2000 | total=1.0146 | CE=0.5967 | SupCon=4.1660 | Ordinal=0.0262
    Episode 600/2000 | total=0.9676 | CE=0.5603 | SupCon=4.0608 | Ordinal=0.0247
    Episode 700/2000 | total=0.9794 | CE=0.5738 | SupCon=4.0439 | Ordinal=0.0248
    Episode 800/2000 | total=0.8768 | CE=0.4778 | SupCon=3.9799 | Ordinal=0.0192
    Episode 900/2000 | total=0.8734 | CE=0.4757 | SupCon=3.9674 | Ordinal=0.0198
    Episode 100

...


H8 OUTER FOLD 4
Training students: 66
Test students: 16

Training class distribution:
label
Engage              6111
Disengage           1623
Highly Engage        815
Highly Disengage     237
Name: count, dtype: int64

H8 meta-training (2000 episodes)...
    Episode 100/2000 | total=2.3597 | CE=1.8569 | SupCon=4.9897 | Ordinal=0.0777
    Episode 200/2000 | total=1.2557 | CE=0.8220 | SupCon=4.3194 | Ordinal=0.0348
    Episode 300/2000 | total=1.1940 | CE=0.7716 | SupCon=4.2066 | Ordinal=0.0350
    Episode 400/2000 | total=1.0337 | CE=0.6175 | SupCon=4.1469 | Ordinal=0.0286
    Episode 500/2000 | total=1.0037 | CE=0.5902 | SupCon=4.1220 | Ordinal=0.0266
    Episode 600/2000 | total=0.9904 | CE=0.5810 | SupCon=4.0820 | Ordinal=0.0253
    Episode 700/2000 | total=0.9479 | CE=0.5397 | SupCon=4.0715 | Ordinal=0.0223
    Episode 800/2000 | total=0.9282 | CE=0.5246 | SupCon=4.0254 | Ordinal=0.0207
    Episode 900/2000 | total=0.9200 | CE=0.5153 | SupCon=4.0359 | Ordinal=0.0217
    Episode 100

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics


H8 OUTER FOLD 5
Training students: 66
Test students: 16

Training class distribution:
label
Engage              6568
Disengage           1727
Highly Engage        912
Highly Disengage     270
Name: count, dtype: int64

H8 meta-training (2000 episodes)...
    Episode 100/2000 | total=2.2785 | CE=1.7753 | SupCon=4.9976 | Ordinal=0.0677
    Episode 200/2000 | total=1.3380 | CE=0.8885 | SupCon=4.4742 | Ordinal=0.0408
    Episode 300/2000 | total=1.1674 | CE=0.7415 | SupCon=4.2413 | Ordinal=0.0342
    Episode 400/2000 | total=1.0816 | CE=0.6649 | SupCon=4.1526 | Ordinal=0.0285
    Episode 500/2000 | total=1.0168 | CE=0.6046 | SupCon=4.1091 | Ordinal=0.0249
    Episode 600/2000 | total=1.0016 | CE=0.5900 | SupCon=4.1028 | Ordinal=0.0264
    Episode 700/2000 | total=1.0172 | CE=0.6073 | SupCon=4.0854 | Ordinal=0.0269
    Episode 800/2000 | total=0.9285 | CE=0.5230 | SupCon=4.0431 | Ordinal=0.0223
    Episode 900/2000 | total=0.8934 | CE=0.4920 | SupCon=4.0042 | Ordinal=0.0197
    Episode 100

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics


H8: LABEL-AWARE PROTOTYPE + SUPERVISED CONTRASTIVE + ORDINAL HYPERNETWORK RESULTS
support_protocol  k_shot  n_students  total_query_clips  mean_accuracy  sd_accuracy  mean_balanced_accuracy  sd_balanced_accuracy  mean_macro_f1  sd_macro_f1  mean_majority_accuracy  mean_majority_balanced_accuracy  mean_majority_macro_f1  mean_prototype_classes_available
           early       5          82              11016       0.631855     0.169210                0.458564              0.153554       0.334533     0.091844                0.708270                         0.307927                0.205566                          1.902439
           early      10          82              10606       0.630984     0.169972                0.460839              0.152275       0.328719     0.089004                0.710367                         0.315041                0.205928                          2.231707
           early      20          82               9786       0.630360     0.175150               